# درس ۰۷ - الگوی طراحی برنامه‌ریزی

این دفترچه الگوی طراحی **برنامه‌ریزی** برای عامل‌های هوش مصنوعی را با استفاده از چارچوب عامل مایکروسافت نشان می‌دهد.  
شما خواهید آموخت چگونه درخواست‌های پیچیده سفر را به وظایف فرعی ساختار یافته تقسیم کنید، آن‌ها را به عامل‌های متخصص اختصاص دهید،  
و طرح حاصل را اجرا کنید — همه با استفاده از خروجی ساختار یافته که توسط مدل‌های Pydantic پشتیبانی می‌شود.


## راه‌اندازی


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## تجزیه وظیفه

تجزیه وظیفه هسته‌ی الگوی طراحی برنامه‌ریزی است. به جای اینکه یک عامل واحد مسئول رسیدگی به یک درخواست پیچیده به صورت انتها به انتها باشد، مسئله را به **زیروظایف** کوچکتر و تعریف‌شده‌ای تقسیم می‌کنیم. هر زیروظیفه به یک عامل متخصص (مثلاً پروازها، هتل‌ها، فعالیت‌ها) با اولویت‌ها و ترتیب وابستگی واضح اختصاص داده می‌شود.

این رویکرد چندین مزیت ارائه می‌دهد:
- **وضوح**: هر زیروظیفه دارای یک مسئولیت واحد است.
- **موازی‌سازی**: زیروظایف مستقل می‌توانند به صورت همزمان اجرا شوند.
- **قابلیت اطمینان**: شکست‌ها محدود به زیروظایف فردی هستند.
- **ردیابی بودجه**: هزینه‌ها برای هر زیروظیفه تخمین زده شده و جمع‌بندی می‌شوند.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ایجاد یک عامل برنامه‌ریزی با خروجی ساختاریافته

عامل برنامه‌ریزی به عنوان یک **هماهنگ‌کننده میز پذیرش** عمل می‌کند. با دریافت یک درخواست سفر در سطح بالا، یک `TravelPlan` ساختاریافته تولید می‌کند — درخواست را به زیرکارها تقسیم می‌کند، اولویت‌ها را تعیین می‌کند، و وابستگی‌ها را شناسایی می‌کند تا یک کارمند پذیرش یا لایه اجرا بتواند کار را انجام دهد.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## اجرای یک برنامه با ابزارهای تخصصی

پس از اینکه کارمند پذیرش یک برنامه ساختارمند تولید کرد، **نماینده دربان** آن را اجرا می‌کند.  
هر ابزار تخصصی یک دسته از زیرکارها را مدیریت می‌کند (پروازها، هتل‌ها، فعالیت‌ها). دربان  
زیرکارهای برنامه را به ترتیب وابستگی طی می‌کند و هر کدام را به ابزار مناسب ارسال می‌کند.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## خلاصه

در این درس شما با **الگوی طراحی برنامه‌ریزی** برای عامل‌های هوش مصنوعی آشنا شدید:

1. **تجزیه وظیفه** — یک عامل برنامه‌ریز پذیرش وظیفه‌ی سفر پیچیده را به زیروظایف ساختارمند تقسیم می‌کند با استفاده از مدل‌های Pydantic و هر کدام را به یک عامل متخصص با اولویت‌ها و وابستگی‌ها اختصاص می‌دهد.
2. **خروجی ساختارمند** — با ارسال `response_format` عامل یک شی validated `TravelPlan` به جای متن آزاد بازمی‌گرداند که پردازش‌های بعدی را قابل اعتماد می‌سازد.
3. **اجرای برنامه** — یک عامل دربان با استفاده از ابزارهای متخصص (`book_flight`، `reserve_hotel`، `book_activity`) برای انجام برنامه و گزارش نتایج، در میان زیروظایف تکرار می‌کند.

این الگو *چه کاری انجام شود* (برنامه‌ریزی) را از *چگونگی انجام آن* (اجرا) جدا می‌کند که باعث می‌شود عامل‌ها ماژولارتر، قابل تست‌تر و آسان‌تر برای توسعه باشند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:  
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما برای دقت تلاش می‌کنیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان بومی خود به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، توصیه می‌شود از ترجمه حرفه‌ای انسانی استفاده شود. ما مسئول هیچ‌گونه سوءتفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه نیستیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
